# Chroma + Semantic Kernel (.NET C#)

End-to-end: ingest PDFs -> store embeddings in Chroma -> retrieve -> chat.

Prereqs: Run the repo task "Setup: Jupyter + .NET Interactive" or execute `scripts/setup-jupyter.ps1`. In Jupyter, select the `.NET (C#)` kernel.


In [5]:
#r "nuget: Microsoft.SemanticKernel, 1.65.*"
#r "nuget: Microsoft.SemanticKernel.Connectors.OpenAI, 1.65.*"
#r "nuget: Microsoft.SemanticKernel.Connectors.Chroma, 1.65.0-*"
#r "nuget: UglyToad.PdfPig, 1.7.0-*"

using System;
using System.IO;
using System.Linq;
using System.Net.Http;
using System.Text;
using System.Threading;
using System.Threading.Tasks;
using System.Collections.Generic;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using Microsoft.SemanticKernel.Embeddings;
using Microsoft.SemanticKernel.Memory;
using Microsoft.SemanticKernel.Connectors.Chroma;
using UglyToad.PdfPig;


Error: Microsoft.SemanticKernel version 1.65.* cannot be added because version 1.65.0 was added previously.

In [7]:
// Configuration (adjust as needed or set environment variables)
string Env(string name, string fallback) { var v = Environment.GetEnvironmentVariable(name); return string.IsNullOrWhiteSpace(v) ? fallback : v; }

var llmBaseUrl = Env("RAG__LLM_BASE_URL", "http://127.0.0.1:1234/v1");
var chatModel = Env("RAG__CHAT_MODEL", "meta-llama-3.1-8b-instruct");
var embedModel = Env("RAG__EMBED_MODEL", "text-embedding-nomic-embed-text-v2");
var apiKey = Env("RAG__LLM_API_KEY", "nokey");
var chromaUrl = Env("RAG__CHROMA_URL", "http://localhost:8000");
var collection = Env("RAG__COLLECTION", "my-document-collection");

Console.WriteLine($"LLM base: {llmBaseUrl}");
Console.WriteLine($"Chat model: {chatModel}");
Console.WriteLine($"Embed model: {embedModel}");
Console.WriteLine($"Chroma: {chromaUrl}");
Console.WriteLine($"Collection: {collection}");


LLM base: http://127.0.0.1:1234/v1
Chat model: meta-llama-3.1-8b-instruct
Embed model: text-embedding-nomic-embed-text-v2
Chroma: http://localhost:8000
Collection: my-document-collection


In [8]:
// Build Semantic Kernel services similar to Program.cs
var builder = Kernel.CreateBuilder();
var http = new HttpClient { BaseAddress = new Uri(llmBaseUrl) };
var embBase = llmBaseUrl;
var embUrlWithSlash = embBase.EndsWith("/") ? embBase : embBase + "/";
var httpEmb = new HttpClient { BaseAddress = new Uri(embUrlWithSlash) };

builder.AddOpenAIChatCompletion(modelId: chatModel, apiKey: apiKey, httpClient: http);
builder.AddOpenAIEmbeddingGenerator(modelId: embedModel, apiKey: apiKey, httpClient: httpEmb);
builder.AddOpenAITextEmbeddingGeneration(modelId: embedModel, apiKey: apiKey, httpClient: httpEmb); // legacy fallback

var kernel = builder.Build();
var chroma = new ChromaMemoryStore(chromaUrl);
ISemanticTextMemory memory = new SemanticTextMemory(chroma, kernel.GetRequiredService<IEmbeddingGenerator<string, Embedding<float>>>());
Console.WriteLine("Kernel + Chroma ready.");


Error: (13,18): error SKEXP0020: 'Microsoft.SemanticKernel.Connectors.Chroma.ChromaMemoryStore' is for evaluation purposes only and is subject to change or removal in future updates. Suppress this diagnostic to proceed.
(13,14): error SKEXP0020: 'Microsoft.SemanticKernel.Connectors.Chroma.ChromaMemoryStore.ChromaMemoryStore(string, Microsoft.Extensions.Logging.ILoggerFactory?)' is for evaluation purposes only and is subject to change or removal in future updates. Suppress this diagnostic to proceed.
(14,1): error SKEXP0001: 'Microsoft.SemanticKernel.Memory.ISemanticTextMemory' is for evaluation purposes only and is subject to change or removal in future updates. Suppress this diagnostic to proceed.
(9,1): error SKEXP0010: 'Microsoft.SemanticKernel.OpenAIKernelBuilderExtensions.AddOpenAIEmbeddingGenerator(Microsoft.SemanticKernel.IKernelBuilder, string, string, string?, int?, string?, System.Net.Http.HttpClient?)' is for evaluation purposes only and is subject to change or removal in

In [ ]:
// Utility: Check Chroma health
using var healthClient = new HttpClient { BaseAddress = new Uri(chromaUrl) };
try {
  var r = await healthClient.GetAsync("/api/v1/heartbeat");
  Console.WriteLine("Chroma heartbeat: " + (int)r.StatusCode);
} catch (Exception ex) { Console.WriteLine("Chroma heartbeat failed: " + ex.Message); }


In [ ]:
// Ingestion helpers (standalone, similar to RAGWorkshop.DataIngestion)
string ExtractTextFromPdf(string filePath)
{
    var sb = new StringBuilder();
    using var doc = PdfDocument.Open(filePath);
    foreach (var page in doc.GetPages()) sb.AppendLine(page.Text);
    return sb.ToString();
}

IEnumerable<string> ChunkText(string text, int chunkSize = 2000, int overlap = 200)
{
    if (string.IsNullOrWhiteSpace(text)) yield break;
    var len = text.Length; int start = 0;
    if (chunkSize <= 0) chunkSize = 2000; if (overlap < 0) overlap = 0;
    while (start < len) {
        var end = Math.Min(start + chunkSize, len);
        yield return text.Substring(start, end - start);
        if (end == len) yield break;
        start = Math.Max(0, end - overlap);
    }
}

async Task IngestPdfAsync(string filePath, ISemanticTextMemory mem, string collectionName)
{
    var text = ExtractTextFromPdf(filePath);
    var i = 0;
    foreach (var chunk in ChunkText(text))
    {
        var id = $"{Path.GetFileName(filePath)}-{i++}";
        await mem.SaveInformationAsync(collectionName, chunk, id);
    }
}

Console.WriteLine("Ingestion helpers ready.");


In [ ]:
// Ingest a single PDF file (edit the path)
var pdfPath = @"C:\path\to\file.pdf"; // TODO: change
if (File.Exists(pdfPath)) {
  Console.WriteLine($"Ingesting {pdfPath} into '{collection}' ...");
  await IngestPdfAsync(pdfPath, memory, collection);
  Console.WriteLine("Done.");
} else { Console.WriteLine("Skip: set pdfPath to an existing PDF."); }


In [ ]:
// Or: Ingest all PDFs under a folder recursively (edit the folder)
var folder = @"C:\path\to\pdfs"; // TODO: change
if (Directory.Exists(folder)) {
  var pdfs = Directory.EnumerateFiles(folder, "*.pdf", SearchOption.AllDirectories).ToList();
  Console.WriteLine($"Found {pdfs.Count} PDF(s). Ingesting into '{collection}' ...");
  foreach (var f in pdfs) { await IngestPdfAsync(f, memory, collection); Console.WriteLine("  + " + f); }
  Console.WriteLine("Done.");
} else { Console.WriteLine("Skip: set folder to an existing directory."); }


In [ ]:
// Retrieve relevant chunks for a question
var question = "What does the document say about architecture?";
var sb = new StringBuilder();
await foreach (var item in memory.SearchAsync(collection, question, limit: 3, minRelevanceScore: 0.75))
{ sb.AppendLine(item.Metadata.Text); }
Console.WriteLine("Context:\n---\n" + sb.ToString());


In [ ]:
// Chat with retrieved context
var chat = kernel.GetRequiredService<IChatCompletionService>();
var history = new ChatHistory();
history.AddSystemMessage("You are a helpful AI assistant answering questions based on the provided context.");
var ctx = new StringBuilder();
await foreach (var item in memory.SearchAsync(collection, question, limit: 3, minRelevanceScore: 0.75)) ctx.AppendLine(item.Metadata.Text);
var userPrompt = $@"Context:\n---\n{ctx}\n---\n\nQuestion: {question}";
history.AddUserMessage(userPrompt);
var response = await chat.GetChatMessageContentAsync(history, kernel: kernel);
Console.WriteLine("AI > " + response.Content);
